# 모기 비행 궤적 예측 AI 경진대회 — Private 2위(0.703151) 통합 솔루션 코드

> 40ms 간격 11개 시점의 3D 좌표로 마지막 관측 **+80ms** 좌표를 예측. 지표 **R-Hit@1cm**(1cm 이내 hit 비율, 높을수록 좋음).

이 노트북은 솔루션 전체를 **하나의 코드로 통합**한 것입니다. 모델 정의는 실제 제출 코드(`src/*.py`)에서 그대로 가져왔습니다.

| 구분 | 점수 | 비고 |
|---|---:|---|
| 시작 베이스라인 (candidate-selection) | 0.6306 | — |
| Kalman 잔차 NN 풀 | LB 0.6888 | 멤버 corr ~0.99 plateau |
| + Neural ODE | LB 0.6912 | 1차 돌파 |
| + Frenet/control-head | LB 0.697 | 2차 돌파 |
| + CREE 회전물리 (수동 α 주입) | LB 0.7016→**0.7022** | 3차 돌파 |
| **최종 v157 (Public 0.7022)** | **Private 0.703151** | **2위** |

---
## 핵심 아이디어 — '직교 메커니즘 다양성'의 앙상블

이 대회의 본질적 난점은 **데이터 천장**이었습니다. 40여 개 모델을 쌓아도 LB 0.6888에서 막혔는데, 원인은 전부 같은 **Kalman 잔차 base** 위에서 학습돼 예측이 **상관계수 ~0.99**로 묶였기 때문입니다. 앙상블 이득은 멤버 간 **직교성**에서 나오므로, 같은 메커니즘의 변종은 아무리 많아도 새 정보가 없습니다.

점수를 움직인 모든 돌파는 **근본적으로 다른 예측 메커니즘(paradigm)을 새로 도입**한 순간이었습니다:

```
                  ┌─ Pool A: Kalman 잔차 프레임 (BiGRU/TCN/Transformer/MDN)
   DE 블렌드 ─────┼─ Pool B: Neural ODE (위치+속도 6D 상태, RK4 적분)
   (base, OOF      ├─ Pool C: Frenet 3D-프레임 ODE + control-head 닫힌형 적분
    0.6831)        └─ (각 paradigm의 boundary refinement 변종)
        │
        │   v157 = (1−α)·base + α·CREE_ens3      ← 최종 수동 주입 (α=0.40, 0.45)
        ▼
   CREE 회전물리 3-앙상블 (Rodrigues 회전 turn-rate 물리, base와 2.82mm 직교)
```

### 결정적 인사이트 — "OOF는 직교 멤버의 LB 프록시가 아니다"
- OOF 최고 블렌드(`v148blend`, OOF 0.6831, CREE weight 0.082) → LB 0.6996
- OOF가 **더 낮은** 블렌드(raw CREE 25% **수동 주입**, OOF 0.6808) → LB **0.7016**
- 즉 **더 낮은 OOF + 더 직교한 변종이 LB를 +0.0020 이긴다.** 1cm hit 지표는 OOF blend CV가 못 잡는 다양성을 보상한다. α를 0.25→0.30→0.40 으로 키우며 Public 0.7016→0.7020→0.7022 단조 상승.

> **규칙 준수 — 회전물리(CREE) 멤버 출처**: 내부 코드네임 `CREE`는 **동명의 본 대회 참가자와 무관**합니다. 대회 기간 중 Dacon **코드 공유 게시판에 공개되었던** 회전 기반 turn-rate 물리 baseline(HyperPhysics 계열)의 모델 구조를 참고했습니다(규칙 8조 B항 공개 코드 공유 허용). 해당 게시물은 **현재 삭제되어 링크 제시 불가**하나 메커니즘은 교과서적 물리(Rodrigues 회전 + EMA 필터)입니다. **공개 모델 구조 코드를 포팅(구조 보존)** 해 우리 5-fold CV·데이터에 연결했고, **가중치 차용 없이 train만으로 from-scratch 학습**했습니다. test 학습·외부데이터·원격 API 모두 미사용, 시드 고정 재현 가능합니다.

---
## §1. 환경 · 데이터 · 공통 코어

모든 paradigm이 공유하는 토대: **Kalman CV base 예측**, **canonical local frame**(마지막 속도 벡터로 yaw 정렬), **scalar features**(속도/가속도/jerk/noise 등), **시퀀스 텐서**. (출처: `src/v23_train.py`)

> 데이터는 `data/train/`, `data/test/`, `data/train_labels.csv`, `data/sample_submission.csv` 에 배치. 전체 학습은 GPU 권장(멤버당 15~30분).

In [ ]:
from __future__ import annotations
import gc, glob, os, random, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from scipy.signal import savgol_filter
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

# --- 경로: 노트북(notebooks/) 기준 프로젝트 루트 탐색 ---
def find_root(start: Path) -> Path:
    p = start.resolve()
    for q in [p, *p.parents]:
        if (q / 'submissions' / 'inputs').exists() or (q / 'data').exists():
            return q
    return p

ROOT = find_root(Path.cwd())
DATA_DIR = ROOT / 'data'
CACHE_DIR = ROOT / 'data' / 'cache'; CACHE_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.environ['PYTHONHASHSEED'] = '0'; random.seed(0); np.random.seed(0); torch.manual_seed(0)
print('ROOT =', ROOT, '| device =', device)

**상수 + 데이터 로딩**

In [ ]:
DT = 0.040
T_PRED = 0.080
T_OBS = np.arange(-400, 1, 40) / 1000.0


def load_data() -> Tuple[np.ndarray, np.ndarray, np.ndarray, pd.DataFrame]:
    cache = CACHE_DIR / "xtrain_xtest.npz"
    train_files = sorted(glob.glob(str(DATA_DIR / "train" / "*.csv")))
    test_files  = sorted(glob.glob(str(DATA_DIR / "test" / "*.csv")))
    labels = pd.read_csv(DATA_DIR / "train_labels.csv")
    sub    = pd.read_csv(DATA_DIR / "sample_submission.csv")
    train_ids = [os.path.splitext(os.path.basename(f))[0] for f in train_files]
    y_train = labels.set_index("id").loc[train_ids][["x", "y", "z"]].values.astype(np.float64)

    if cache.exists():
        nc = np.load(cache)
        X_train, X_test = nc["X_train"], nc["X_test"]
        print(f"[data] cache 로드: X_train {X_train.shape}")
    else:
        def _read(f): return pd.read_csv(f)[["x", "y", "z"]].values
        def _load(files, desc, workers=8):
            arrays = [None] * len(files)
            with ThreadPoolExecutor(max_workers=workers) as ex:
                futs = {ex.submit(_read, f): i for i, f in enumerate(files)}
                for fu in tqdm(futs, desc=desc, total=len(futs)):
                    arrays[futs[fu]] = fu.result()
            return np.stack(arrays, axis=0).astype(np.float64)
        X_train = _load(train_files, "train")
        X_test  = _load(test_files, "test")
        np.savez(cache, X_train=X_train, X_test=X_test)
        print(f"[data] cache 저장: {cache}")

    assert X_train.shape[0] == y_train.shape[0] == 10000
    return X_train, X_test, y_train, sub

**Kalman CV base 예측** — σ_obs=0.3mm, σ_proc=1.0 (상수속도 모델)

In [ ]:
def kalman_predict(X, sigma_obs=0.30e-3, sigma_proc=1.0, P0=1.0):
    N, T, _ = X.shape
    F = np.array([[1, DT], [0, 1]])
    F_pred = np.array([[1, T_PRED], [0, 1]])
    Q = sigma_proc ** 2 * np.array([[DT**4/4, DT**3/2], [DT**3/2, DT**2]])
    R = sigma_obs ** 2
    pred = np.zeros((N, 3))
    for j in range(3):
        z_all = X[:, :, j]
        state = np.zeros((N, 2)); state[:, 0] = z_all[:, 0]
        P = np.eye(2) * P0
        for t in range(1, T):
            state = state @ F.T
            P = F @ P @ F.T + Q
            innov = z_all[:, t] - state[:, 0]
            S = P[0, 0] + R; K = P[:, 0] / S
            state = state + np.outer(innov, K)
            P = P - np.outer(K, P[0, :])
        pred[:, j] = (state @ F_pred.T)[:, 0]
    return pred


def get_kalman(X_train, X_test):
    cache = CACHE_DIR / "kalman.npz"
    if cache.exists():
        kc = np.load(cache)
        print("[kalman] cache 로드")
        return kc["kalman_train"], kc["kalman_test"], kc["kalman_train_alt"]
    print("[kalman] 계산 중 (10~30초)…")
    k_tr = kalman_predict(X_train, sigma_obs=0.30e-3, sigma_proc=1.0)
    k_te = kalman_predict(X_test,  sigma_obs=0.30e-3, sigma_proc=1.0)
    k_tr_alt = kalman_predict(X_train, sigma_obs=1.0e-3, sigma_proc=1.0)
    np.savez(cache, kalman_train=k_tr, kalman_test=k_te, kalman_train_alt=k_tr_alt)
    print(f"[kalman] cache 저장: {cache}")
    return k_tr, k_te, k_tr_alt

**Scalar features** — 속도·가속도·jerk·직진성·turn·noise(poly2/savgol/LOO-spline) 등 + log1p

In [ ]:
def noise_poly2(X):
    V = np.vander(T_OBS, 3, increasing=False)
    out = np.zeros(X.shape[0])
    for j in range(3):
        coef = np.linalg.lstsq(V, X[:, :, j].T, rcond=None)[0]
        out += (X[:, :, j] - (V @ coef).T).std(axis=1)
    return out / 3


def noise_savgol(X, w=5, p=2):
    Xs = savgol_filter(X, window_length=w, polyorder=p, axis=1)
    return (X - Xs).std(axis=1).mean(axis=-1)


def noise_loo_subset(X, sample_idx):
    sav = noise_savgol(X)
    out = sav.copy()
    idx_all = np.arange(len(T_OBS))
    for i in tqdm(sample_idx, desc="LOO spline subset"):
        s = 0.0
        for k in range(1, len(T_OBS) - 1):
            mask = idx_all != k
            for j in range(3):
                cs = CubicSpline(T_OBS[mask], X[i, mask, j])
                s += (X[i, k, j] - cs(T_OBS[k])) ** 2
        out[i] = np.sqrt(s / ((len(T_OBS) - 2) * 3))
    return out


def cos_safe(a, b):
    na = np.linalg.norm(a, axis=-1); nb = np.linalg.norm(b, axis=-1)
    return np.clip((a * b).sum(-1) / np.maximum(na * nb, 1e-12), -1, 1)


def build_scalar_feats(X, noise_p, noise_s, noise_l=None):
    delta_ = np.diff(X, axis=1)
    v_ = delta_ / DT
    a_ = np.diff(v_, axis=1) / DT
    jerk_ = np.diff(a_, axis=1) / DT
    sp_ = np.linalg.norm(v_, axis=-1)
    ac_ = np.linalg.norm(a_, axis=-1)
    jk_ = np.linalg.norm(jerk_, axis=-1)
    v_l = v_[:, -1, :]; a_l = a_[:, -1, :]
    a_r = a_[:, -3:, :].mean(axis=1)
    nd_vec = X[:, -1] - X[:, 0]; nd = np.linalg.norm(nd_vec, axis=-1)
    pl = np.linalg.norm(delta_, axis=-1).sum(axis=1)
    straight = np.where(pl > 1e-12, nd / np.maximum(pl, 1e-12), 0.0)
    turn = cos_safe(v_l, v_[:, :-1, :].mean(axis=1))
    if noise_l is None: noise_l = noise_s
    return pd.DataFrame({
        "mean_speed": sp_.mean(1), "max_speed": sp_.max(1),
        "speed_std": sp_.std(1), "mean_acc": ac_.mean(1),
        "max_acc": ac_.max(1), "max_jerk": jk_.max(1),
        "straightness": straight, "net_disp": nd,
        "turn_cos": turn, "|v_last|": np.linalg.norm(v_l, axis=-1),
        "|a_last|": np.linalg.norm(a_l, axis=-1),
        "|a_recent|": np.linalg.norm(a_r, axis=-1),
        "jerk_last": jk_[:, -1], "jerk_recent": jk_[:, -3:].mean(1),
        "noise_poly2": noise_p, "noise_savgol": noise_s, "noise_loo": noise_l,
        "hard_turn": (turn < 0.5).astype(np.float32),
        "high_speed": (np.linalg.norm(v_l, axis=-1) > 1.0).astype(np.float32),
        "high_acc": (ac_.max(axis=1) > 15).astype(np.float32),
        "log_max_acc": np.log1p(ac_.max(1)),
    })


def get_scalar_feats(X_train, X_test, mode_cfg, mode_name):
    cache = CACHE_DIR / f"noise_{mode_name}.npz"
    if cache.exists():
        nc = np.load(cache)
        np_tr, ns_tr, nl_tr = nc["noise_p"], nc["noise_s"], nc["noise_l"]
        np_te, ns_te = nc["noise_p_test"], nc["noise_s_test"]
        print("[noise] cache 로드")
    else:
        print("[noise] 계산 중…")
        np_tr = noise_poly2(X_train); np_te = noise_poly2(X_test)
        ns_tr = noise_savgol(X_train); ns_te = noise_savgol(X_test)
        if mode_cfg["loo_sample"] is None:
            loo_idx = np.arange(X_train.shape[0])
        else:
            rng = np.random.RandomState(0)
            loo_idx = rng.choice(X_train.shape[0], size=mode_cfg["loo_sample"], replace=False)
        nl_tr = noise_loo_subset(X_train, loo_idx)
        np.savez(cache, noise_p=np_tr, noise_s=ns_tr, noise_l=nl_tr,
                  noise_p_test=np_te, noise_s_test=ns_te)
        print(f"[noise] cache 저장: {cache}")

    scal_tr = build_scalar_feats(X_train, np_tr, ns_tr, nl_tr)
    scal_te = build_scalar_feats(X_test,  np_te, ns_te)
    LOG_COLS = ["mean_speed","max_speed","speed_std","mean_acc","max_acc","max_jerk",
                "net_disp","|v_last|","|a_last|","|a_recent|","jerk_last","jerk_recent",
                "noise_poly2","noise_savgol","noise_loo"]
    for c in LOG_COLS:
        scal_tr[c] = np.log1p(scal_tr[c])
        scal_te[c] = np.log1p(scal_te[c])
    return scal_tr.values.astype(np.float32), scal_te.values.astype(np.float32)

**Canonical frame(yaw 정렬) + 시퀀스 텐서**

In [ ]:
def yaw_angle(v):
    return np.arctan2(v[:, 1], v[:, 0])


def rotate_xy(vec, theta):
    c = np.cos(theta); s = np.sin(theta)
    return np.stack([vec[:,0]*c + vec[:,1]*s,
                      -vec[:,0]*s + vec[:,1]*c,
                      vec[:,2]], axis=-1)


def inverse_rotate_xy(vec, theta):
    c = np.cos(theta); s = np.sin(theta)
    return np.stack([vec[:,0]*c - vec[:,1]*s,
                      vec[:,0]*s + vec[:,1]*c,
                      vec[:,2]], axis=-1)


def build_tier3(X):
    disp = np.diff(X, axis=1); v = disp / DT
    speed = np.linalg.norm(v, axis=-1)
    roll = np.stack([speed[:, i:i+3].mean(axis=1) for i in range(8)], axis=1) * DT
    cum = np.concatenate([np.zeros((X.shape[0], 1)),
                           np.cumsum(np.linalg.norm(disp, axis=-1), axis=1)], axis=1)
    return np.concatenate([roll, cum], axis=1).astype(np.float32)


def build_seq(X):
    N = X.shape[0]
    rel = X - X[:, -1:, :]
    disp = np.diff(X, axis=1); v = disp / DT
    v_pad = np.concatenate([np.zeros((N,1,3)), v], axis=1)
    a = np.diff(v, axis=1) / DT
    a_pad = np.concatenate([np.zeros((N,2,3)), a], axis=1)
    return np.concatenate([rel, v_pad, a_pad], axis=-1).astype(np.float32)


def normalize_seq(arr, sc):
    N, T, C = arr.shape
    return sc.transform(arr.reshape(-1, C)).astype(np.float32).reshape(N, T, C)

---
## §2. Pool A — Kalman 잔차 GRU (baseline paradigm)

canonical local frame에서 **Kalman 잔차**를 타깃으로 GRU 학습. 보조 헤드(F: last-obs 잔차, W: 다른 σ Kalman 잔차)와 **combo loss = euclid + 0.3·softhit**. yaw + y-mirror 증강으로 회전 불변. 백본을 BiGRU/TCN/Transformer/MDN으로 바꿔 멤버를 다양화했지만 **이 풀 내부 corr ~0.99 → LB 0.6888 천장**. (출처: `src/v23_train.py`, `src/v77_bigru.py` 등)

**모델 + 손실**

In [ ]:
def loss_euclid(pred, true):
    return torch.sqrt(((pred - true) ** 2).sum(dim=-1) + 1e-12).mean()


def loss_softhit(pred, true, beta=0.002):
    d = torch.sqrt(((pred - true) ** 2).sum(dim=-1) + 1e-12)
    return torch.sigmoid((d - 0.01) / beta).mean()


def loss_combo(pred, true):
    return loss_euclid(pred, true) + 0.3 * loss_softhit(pred, true)


class GRUModelMultiAux(nn.Module):
    def __init__(self, n_channels=9, scal_dim=40, hidden=64, layers=1,
                 fc=128, p=0.2, aux_dims=(3, 3), main_scale_cm=2.0):
        super().__init__()
        self.gru = nn.GRU(n_channels, hidden, num_layers=layers, batch_first=True,
                           dropout=p if layers > 1 else 0)
        self.fc1 = nn.Linear(hidden + scal_dim, fc)
        self.fc2 = nn.Linear(fc, fc // 2)
        self.act = nn.GELU(); self.drop = nn.Dropout(p)
        self.head_main = nn.Linear(fc // 2, 3)
        self.aux_heads = nn.ModuleList([nn.Linear(fc // 2, d) for d in aux_dims])
        self.main_scale = main_scale_cm / 100.0

    def forward(self, seq, scal):
        out, _ = self.gru(seq)
        z = torch.cat([out[:, -1, :], scal], dim=1)
        z = self.act(self.fc1(z)); z = self.drop(z)
        z = self.act(self.fc2(z))
        out_main = torch.tanh(self.head_main(z)) * self.main_scale
        return out_main, [h(z) for h in self.aux_heads]

**5-fold OOF 학습 러너** (Pool A) — 멤버마다 OOF/test 예측을 생성해 블렌드 풀에 적재

In [ ]:
def run_kfold(target_main, target_F, target_W,
              seq_arr, scal_arr, seq_te, scal_te,
              kalman_train, theta_train, theta_test, y_train,
              config, mode_cfg, lambda_F=0.3, lambda_W=0.3, device="cpu"):
    n_folds, n_seeds = mode_cfg["n_folds"], mode_cfg["n_seeds"]
    max_epochs, patience, batch = mode_cfg["max_epochs"], mode_cfg["patience"], mode_cfg["batch"]

    oof_rot = np.zeros((len(target_main), 3))
    fold_mask = np.zeros(len(target_main), dtype=bool)
    test_per_fold = []
    fold_rh = []
    kf = KFold(n_splits=max(n_folds, 2), shuffle=True, random_state=0)
    fold_iter = list(kf.split(np.arange(len(target_main))))
    if n_folds == 1:
        fold_iter = fold_iter[:1]
    t0 = time.time()

    for fi, (tr, va) in enumerate(fold_iter):
        sc_seq = StandardScaler().fit(seq_arr[tr].reshape(-1, seq_arr.shape[2]))
        sc_scal = StandardScaler().fit(scal_arr[tr])
        seq_tr_n = normalize_seq(seq_arr[tr], sc_seq)
        seq_va_n = normalize_seq(seq_arr[va], sc_seq)
        seq_te_n = normalize_seq(seq_te, sc_seq)
        scal_tr_n = sc_scal.transform(scal_arr[tr]).astype(np.float32)
        scal_va_n = sc_scal.transform(scal_arr[va]).astype(np.float32)
        scal_te_n = sc_scal.transform(scal_te).astype(np.float32)

        def T(a): return torch.from_numpy(a).to(device)
        seq_tr_t, scal_tr_t = T(seq_tr_n), T(scal_tr_n)
        tgt_tr_t = T(target_main[tr].astype(np.float32))
        F_tr_t   = T(target_F[tr].astype(np.float32))
        W_tr_t   = T(target_W[tr].astype(np.float32))
        seq_va_t, scal_va_t = T(seq_va_n), T(scal_va_n)
        seq_te_t, scal_te_t = T(seq_te_n), T(scal_te_n)

        seed_val, seed_test = [], []
        for s in range(n_seeds):
            torch.manual_seed(s); np.random.seed(s)
            model = GRUModelMultiAux(
                n_channels=seq_arr.shape[2], scal_dim=scal_arr.shape[1],
                hidden=config["hidden"], layers=config["layers"],
                fc=config["fc"], p=config["p"],
            ).to(device)
            opt = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["wd"])
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

            best_rh, best_state, no_improve = -1.0, None, 0
            n_tr = seq_tr_t.shape[0]
            for ep in range(max_epochs):
                model.train()
                perm = torch.randperm(n_tr)
                for i in range(0, n_tr, batch):
                    idx = perm[i:i+batch]
                    opt.zero_grad()
                    out_main, outs_aux = model(seq_tr_t[idx], scal_tr_t[idx])
                    loss = loss_combo(out_main, tgt_tr_t[idx])
                    loss = loss + lambda_F * loss_euclid(outs_aux[0], F_tr_t[idx])
                    loss = loss + lambda_W * loss_euclid(outs_aux[1], W_tr_t[idx])
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                sched.step()

                model.eval()
                with torch.no_grad():
                    out_va_rot, _ = model(seq_va_t, scal_va_t)
                    out_va_rot = out_va_rot.cpu().numpy()
                out_va = inverse_rotate_xy(out_va_rot, theta_train[va])
                pred = kalman_train[va] + out_va
                rh = float((np.linalg.norm(pred - y_train[va], axis=-1) <= 0.01).mean())
                if rh > best_rh:
                    best_rh = rh
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                    no_improve = 0
                else:
                    no_improve += 1
                if no_improve >= patience: break
                if ep == 0 or (ep + 1) % 5 == 0:
                    print(f"  fold{fi+1} seed{s} ep{ep+1:3d}: rhit={rh:.4f} (best {best_rh:.4f})  "
                          f"[{(time.time()-t0)/60:.1f}m]", flush=True)

            model.load_state_dict(best_state); model.eval()
            with torch.no_grad():
                pv_rot, _ = model(seq_va_t, scal_va_t)
                pt_rot, _ = model(seq_te_t, scal_te_t)
            seed_val.append(pv_rot.cpu().numpy())
            seed_test.append(pt_rot.cpu().numpy())
            del model; gc.collect()

        val_mean_rot = np.mean(seed_val, axis=0)
        test_mean_rot = np.mean(seed_test, axis=0)
        oof_rot[va] = val_mean_rot
        fold_mask[va] = True
        test_per_fold.append(test_mean_rot)

        val_unrot = inverse_rotate_xy(val_mean_rot, theta_train[va])
        pred_pos = kalman_train[va] + val_unrot
        rh_fold = float((np.linalg.norm(pred_pos - y_train[va], axis=-1) <= 0.01).mean())
        fold_rh.append(rh_fold)
        print(f"  ★ fold {fi+1}/{len(fold_iter)}: R-Hit={rh_fold:.4f}  ({(time.time()-t0)/60:.1f}m)", flush=True)

    if fold_mask.sum() == 0:
        oof_rhit = float("nan"); oof_unrot_all = np.zeros_like(target_main)
    else:
        oof_unrot_all = np.zeros_like(target_main)
        oof_unrot_all[fold_mask] = inverse_rotate_xy(oof_rot[fold_mask], theta_train[fold_mask])
        pred = kalman_train[fold_mask] + oof_unrot_all[fold_mask]
        oof_rhit = float((np.linalg.norm(pred - y_train[fold_mask], axis=-1) <= 0.01).mean())

    test_unrot = np.mean([inverse_rotate_xy(rot, theta_test) for rot in test_per_fold], axis=0)
    print(f"  OOF R-Hit (covered {fold_mask.sum()}): {oof_rhit:.4f}  소요 {(time.time()-t0)/60:.1f}m")
    return oof_unrot_all, test_unrot, fold_rh, oof_rhit, fold_mask

---
## §3. Pool B — Neural ODE (1차 돌파, LB 0.6912)

타깃을 Kalman과 무관한 `y − last_obs`로 바꿔 **완전히 다른 base**. **6D 상태(위치·속도)** 에 학습된 감쇠 + neural acceleration field를 두고 80ms 구간을 **RK4 4-eval**로 적분. 단독 OOF는 낮지만(0.66) base 풀과 **L2 ~2.2mm 직교** → DE 블렌더가 ~40% 가중치 부여 → 0.6888 → **0.6912** 돌파. (출처: `src/v120_neural_ode.py`)

**yaw 회전(시퀀스) + mirror 증강**

In [ ]:
def rotate_xy_seq(seq_xyz: np.ndarray, theta: np.ndarray) -> np.ndarray:
    """rotate (N, T, 3) by theta (N,) around z. y_local = -x*sin + y*cos."""
    c = np.cos(theta)[:, None]; s = np.sin(theta)[:, None]
    x_new = seq_xyz[..., 0] * c + seq_xyz[..., 1] * s
    y_new = -seq_xyz[..., 0] * s + seq_xyz[..., 1] * c
    return np.stack([x_new, y_new, seq_xyz[..., 2]], axis=-1).astype(np.float32)


# ============================================================
# Mirror aug (y-flip) — v90/v104 패턴
# ============================================================
def mirror_seq(seq):
    out = seq.copy()
    out[..., 1] *= -1; out[..., 4] *= -1; out[..., 7] *= -1
    return out

def mirror_target(t):
    out = t.copy(); out[..., 1] *= -1; return out


**Neural ODE 모델** — Encoder → latent, dp/dt=v, dv/dt=−damping⊙v+a(p,v,z,speed), RK4 적분

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(dim, dim),
        )
        self.ln = nn.LayerNorm(dim)
    def forward(self, x):
        return self.ln(x + self.net(x))


class AccelField(nn.Module):
    """가속도 필드 a(pos, vel, latent, speed) → R^3."""
    def __init__(self, latent_dim=64, hidden=64):
        super().__init__()
        # pos(3) + vel(3) + latent + speed_scalar(1) = 7 + latent
        self.net = nn.Sequential(
            nn.Linear(3 + 3 + latent_dim + 1, hidden),
            nn.LayerNorm(hidden), nn.GELU(),
            ResBlock(hidden),
            nn.Linear(hidden, 3),
        )

    def forward(self, pos, vel, latent, speed):
        if speed.dim() == 1: speed = speed.unsqueeze(-1)
        x = torch.cat([pos, vel, latent, speed], dim=-1)
        return self.net(x)


class NeuralODEModel(nn.Module):
    """Position-velocity 6D state, RK4 single-step 80ms integration.

    Encoder: flatten(seq local-rotated) + scalar feats → latent z₀
    Dynamics: dp/dt = v ; dv/dt = -damping ⊙ v + a_neural(p,v,z,speed)
    Integration: RK4 (4 evals, dt=80ms)
    Output: final local position + local_bias → rotate back + last_obs + global_bias
    """
    def __init__(self, seq_dim=99, scal_dim=40, latent_dim=64, hidden=64,
                  dt_pred=0.080, n_steps=1):
        super().__init__()
        self.dt_pred = dt_pred
        self.n_steps = n_steps
        in_dim = seq_dim + scal_dim
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, latent_dim),
            nn.LayerNorm(latent_dim), nn.GELU(),
            ResBlock(latent_dim),
            ResBlock(latent_dim),
        )
        self.accel_field = AccelField(latent_dim=latent_dim, hidden=hidden)
        self.learned_damping = nn.Parameter(torch.tensor([0.1, 0.1, 0.1]))
        self.local_bias = nn.Parameter(torch.zeros(3))
        self.global_bias = nn.Parameter(torch.zeros(3))
        self._last_accels = []

    def _ode_deriv(self, pos, vel, latent, speed):
        a = self.accel_field(pos, vel, latent, speed)
        dp = vel
        dv = -self.learned_damping * vel + a
        return dp, dv, a

    def _rk4_step(self, pos, vel, latent, speed, dt):
        dp1, dv1, a1 = self._ode_deriv(pos, vel, latent, speed)
        pos2 = pos + 0.5 * dt * dp1; vel2 = vel + 0.5 * dt * dv1
        dp2, dv2, a2 = self._ode_deriv(pos2, vel2, latent, speed)
        pos3 = pos + 0.5 * dt * dp2; vel3 = vel + 0.5 * dt * dv2
        dp3, dv3, a3 = self._ode_deriv(pos3, vel3, latent, speed)
        pos4 = pos + dt * dp3; vel4 = vel + dt * dv3
        dp4, dv4, a4 = self._ode_deriv(pos4, vel4, latent, speed)
        new_pos = pos + (dt / 6.0) * (dp1 + 2 * dp2 + 2 * dp3 + dp4)
        new_vel = vel + (dt / 6.0) * (dv1 + 2 * dv2 + 2 * dv3 + dv4)
        return new_pos, new_vel, [a1, a2, a3, a4]

    def forward(self, seq_flat, scal, init_vel, speed):
        """Returns predicted local-frame displacement from last obs.

        seq_flat: (B, seq_dim) flattened normalized seq (local-rotated)
        scal:     (B, scal_dim)
        init_vel: (B, 3) initial velocity in local frame
        speed:    (B,) magnitude of init_vel (separate input for stability)
        """
        latent = self.backbone(torch.cat([seq_flat, scal], dim=-1))
        pos = torch.zeros_like(init_vel)
        vel = init_vel
        dt = self.dt_pred / self.n_steps
        accels = []
        for _ in range(self.n_steps):
            pos, vel, ac = self._rk4_step(pos, vel, latent, speed, dt)
            accels.extend(ac)
        self._last_accels = accels
        return pos + self.local_bias

**손실** — huber + softhit + accel 정규화

In [ ]:
def loss_huber(pred, true, delta=0.001):
    return F.huber_loss(pred, true, delta=delta)

def loss_softhit(pred, true, k=300.0, c=0.01):
    d = torch.sqrt(((pred - true) ** 2).sum(-1) + 1e-12)
    return torch.sigmoid((d - c) * k).mean()


def loss_combined(pred, true, accels, w_huber=100.0, w_hit=1.0, w_reg=1e-4):
    huber = loss_huber(pred, true)
    hit = loss_softhit(pred, true)
    reg = 0.0
    if accels:
        reg = sum(a.pow(2).sum(-1).mean() for a in accels) / len(accels)
    return w_hit * hit + w_huber * huber + w_reg * reg, huber.item(), hit.item()

---
## §4. Pool C — Frenet 3D-프레임 + control-head (2차 돌파, LB 0.697)

**Frenet 프레임**: tangent(속도)·normal(가속도)·binormal 으로 만든 완전 3D 직교 프레임. yaw(xy)만 회전하는 v120과 달리 **z 처리가 근본적으로 달라** decorrelation 최대. **control-head**: RK4 대신 NN이 가속도(control)를 출력하고 `p = v₀·T + ½·a·T²` **닫힌형 적분** → 또 다른 오차 구조. conservative 블렌드 OOF 0.6807, 신규 멤버 가중 58% → LB **0.697** (변환률 +0.0165). (출처: `src/v131_paradigm_variants.py`, `src/v135_control_head.py`)

**좌표 프레임(yaw/Frenet) + 프레임 정합 mirror**

In [ ]:
def yaw_frame(X):
    v = (X[:, -1] - X[:, -2]) / DT
    theta = yaw_angle(v)
    c, s = np.cos(theta), np.sin(theta)
    R = np.zeros((X.shape[0], 3, 3))
    # x' = c*x + s*y ; y' = -s*x + c*y ; z'=z   (matches rotate_xy)
    R[:, 0, 0] = c;  R[:, 0, 1] = s
    R[:, 1, 0] = -s; R[:, 1, 1] = c
    R[:, 2, 2] = 1.0
    return R

def frenet_frame(X):
    """tangent=last vel, normal from last accel (Gram-Schmidt), binormal=t×n.
    degenerate(저속/직선) 샘플은 yaw frame으로 fallback."""
    N = X.shape[0]
    v = (X[:, -1] - X[:, -2]) / DT                       # (N,3)
    a = (X[:, -1] - 2 * X[:, -2] + X[:, -3]) / (DT * DT)  # (N,3)
    nv = np.linalg.norm(v, axis=1)
    t = v / np.clip(nv[:, None], 1e-9, None)
    # normal = a - (a·t)t
    a_perp = a - (np.sum(a * t, axis=1, keepdims=True)) * t
    na = np.linalg.norm(a_perp, axis=1)
    n = a_perp / np.clip(na[:, None], 1e-9, None)
    b = np.cross(t, n)
    R = np.stack([t, n, b], axis=1)  # rows = frame axes → R@g = local
    # fallback: degenerate where |v|<1e-4 or |a_perp|<1e-6 → use yaw frame
    Ry = yaw_frame(X)
    bad = (nv < 1e-4) | (na < 1e-6)
    R[bad] = Ry[bad]
    return R

def apply_R_seq(seq, R):
    """seq (N,T,9)=[rel,v,a] 각 3-block에 R 적용 (local = R@vec)."""
    out = np.empty_like(seq)
    for k in range(3):
        blk = seq[..., 3*k:3*k+3]                       # (N,T,3)
        out[..., 3*k:3*k+3] = np.einsum("nij,ntj->nti", R, blk)
    return out.astype(np.float32)

def apply_R_vec(vec, R):       # (N,3) global→local
    return np.einsum("nij,nj->ni", R, vec).astype(np.float32)

def inv_R_vec(vec, R):         # (N,3) local→global  (R^T)
    return np.einsum("nji,nj->ni", R, vec)

def mirror_seq(seq, axis=1):
    """물리적 좌우반사를 local 좌표로 변환한 mirror.
    yaw frame: world-y negate -> local axis=1 (+ vel/accel blocks).
    frenet frame: 좌우반사 M=diag(1,-1,1)이 Frenet-local에서 binormal(z)만 negate
                  -> axis=2. (유도: local=(t·g, n·g, b·g), 반사 후 b_new=-Mb -> z성분 부호반전)"""
    out = seq.copy()
    for blk in range(3):
        out[..., 3 * blk + axis] *= -1
    return out

def mirror_vec(v, axis=1):
    out = v.copy(); out[..., axis] *= -1; return out

**GRU-encoder ODE** (프레임 무관; Frenet/yaw 어디서나 동작)

In [ ]:
class GRUODEModel(nn.Module):
    def __init__(self, seq_channels=9, scal_dim=40, latent_dim=64, hidden=64,
                 dt_pred=0.080, n_steps=1):
        super().__init__()
        self.dt_pred = dt_pred; self.n_steps = n_steps
        self.gru = nn.GRU(seq_channels, latent_dim, num_layers=2, batch_first=True, dropout=0.1)
        self.scal_proj = nn.Sequential(nn.Linear(scal_dim, latent_dim), nn.LayerNorm(latent_dim), nn.GELU())
        self.fuse = nn.Sequential(nn.Linear(2*latent_dim, latent_dim), nn.LayerNorm(latent_dim),
                                  nn.GELU(), ResBlock(latent_dim))
        self.accel_field = AccelField(latent_dim=latent_dim, hidden=hidden)
        self.learned_damping = nn.Parameter(torch.tensor([0.1, 0.1, 0.1]))
        self.local_bias = nn.Parameter(torch.zeros(3))
        self._last_accels = []

    def _ode_deriv(self, pos, vel, latent, speed):
        a = self.accel_field(pos, vel, latent, speed)
        return vel, -self.learned_damping * vel + a, a

    def _rk4(self, pos, vel, latent, speed, dt):
        dp1, dv1, a1 = self._ode_deriv(pos, vel, latent, speed)
        dp2, dv2, a2 = self._ode_deriv(pos + .5*dt*dp1, vel + .5*dt*dv1, latent, speed)
        dp3, dv3, a3 = self._ode_deriv(pos + .5*dt*dp2, vel + .5*dt*dv2, latent, speed)
        dp4, dv4, a4 = self._ode_deriv(pos + dt*dp3, vel + dt*dv3, latent, speed)
        np_ = pos + (dt/6)*(dp1+2*dp2+2*dp3+dp4)
        nv_ = vel + (dt/6)*(dv1+2*dv2+2*dv3+dv4)
        return np_, nv_, [a1, a2, a3, a4]

    def forward(self, seq, scal, init_vel, speed):
        # seq: (B,T,C)
        h = self.gru(seq)[0][:, -1]                     # last hidden (B,latent)
        latent = self.fuse(torch.cat([h, self.scal_proj(scal)], dim=-1))
        pos = torch.zeros_like(init_vel); vel = init_vel
        dt = self.dt_pred / self.n_steps; accels = []
        for _ in range(self.n_steps):
            pos, vel, ac = self._rk4(pos, vel, latent, speed, dt); accels.extend(ac)
        self._last_accels = accels
        return pos + self.local_bias

**control-head** — 닫힌형 적분 멤버 (`p = v_scale·v₀·T + ½·a·T²` [+ ⅙·j·T³])

In [ ]:
class ControlHead(nn.Module):
    """seq+scal -> latent -> control(accel[, jerk]); analytic integrate to +80ms disp."""
    def __init__(self, seq_dim, scal_dim, latent_dim=64, order="accel", T=0.080):
        super().__init__()
        self.T = T; self.order = order
        self.backbone = nn.Sequential(
            nn.Linear(seq_dim + scal_dim, latent_dim), nn.LayerNorm(latent_dim), nn.GELU(),
            ResBlock(latent_dim), ResBlock(latent_dim))
        nc = 3 if order == "accel" else 6  # accel | accel+jerk
        self.head = nn.Linear(latent_dim, nc)
        self.local_bias = nn.Parameter(torch.zeros(3))
        self.v_scale = nn.Parameter(torch.ones(1))   # learned trust on init velocity

    def forward(self, seq_flat, scal, init_vel, speed):
        z = self.backbone(torch.cat([seq_flat, scal], dim=-1))
        c = self.head(z); T = self.T
        a = c[:, :3]
        disp = self.v_scale * init_vel * T + 0.5 * a * (T * T)
        if self.order == "jerk":
            j = c[:, 3:6]; disp = disp + (1.0 / 6.0) * j * (T ** 3)
        return disp + self.local_bias

---
## §5. CREE 회전물리 멤버 (최종 돌파, LB 0.7016 → 0.7022)

**공개 Dacon 코드공유 baseline(HyperPhysics, 회전 기반 turn-rate 물리)** 을 우리 5-fold OOF 파이프라인에 포팅. Rodrigues 회전으로 속도 벡터를 회전 + 학습된 angular velocity(omega) + EMA 속도/가속도 필터 + world-up 프레임. 우리 RK4 적분 계열과 메커니즘이 완전히 달라 **base와 2.82mm 직교**(우리 frenet 멤버끼리는 0.6mm). `dirnet(seed42) + dirnet(seed1) + 3step-heading` **3-앙상블**로 강화 — 내부 분산만 줄이고 교차-paradigm 직교성은 보존. (출처: `src/v148_cree_xy2.py`)

> 출처·규칙 준수는 §0 상단 주석 참조. 아래는 포팅한 모델 구조(가중치 차용 없이 train만으로 학습).

**Sliding-window 데이터셋 + EMA + soft-hit 손실 + 회전-프레임 피처 추출**

In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, X, y, min_win=3, mode="extended", device="cpu", targets_ext=None):
        Xt = torch.tensor(X, dtype=torch.float32); yt = torch.tensor(y, dtype=torch.float32)
        windows = []
        for i in range(len(X)):
            targets = (targets_ext or [4,5,6,7,8,9,10,12]) if mode == "extended" else [12, 10]
            for target_idx in targets:
                end_idx = target_idx - 2
                max_w = end_idx + 2 if mode == "extended" else (12 if target_idx == 12 else 10)
                for w in range(min_win, max_w):
                    windows.append((i, w, target_idx))
        Xl, yl = [], []
        for i, w, target_idx in windows:
            Xo = Xt[i]; end_idx = target_idx - 2
            pts = Xo[end_idx - w + 1: end_idx + 1]
            target = yt[i] if target_idx == 12 else Xo[target_idx]
            if w < 11:
                v0 = pts[1] - pts[0]; n_pad = 11 - w
                js = torch.arange(n_pad, 0, -1, dtype=torch.float32)
                pad = pts[0:1] - js.unsqueeze(1) * v0.unsqueeze(0)
                Xp = torch.cat([pad, pts], dim=0)
            else:
                Xp = pts.clone()
            Xl.append(Xp); yl.append(target)
        self.X_all = torch.stack(Xl).to(device); self.y_all = torch.stack(yl).to(device)
        diffs = self.X_all[:, 1:] - self.X_all[:, :-1]
        n1 = diffs[:, 1:].norm(dim=2).clamp(min=1e-8); n2 = diffs[:, :-1].norm(dim=2).clamp(min=1e-8)
        cos_t = ((diffs[:, 1:] * diffs[:, :-1]).sum(dim=2) / (n1 * n2)).clamp(-1, 1)
        theta_last = torch.acos(cos_t[:, -1])
        self.theta_weights = (1.0 + 4.0 * (theta_last / 1.0).clamp(0, 1)).cpu().numpy()
    def __len__(self): return len(self.X_all)
    def __getitem__(self, idx): return self.X_all[idx], self.y_all[idx]

def _ema_va_local(diffs_local, alpha, beta):
    B, T, _ = diffs_local.shape; one_m_a = 1.0 - alpha; one_m_b = 1.0 - beta
    vs = diffs_local.new_empty(B, T, 3); v = diffs_local[:, 0]; vs[:, 0] = v
    for t in range(1, T):
        v = alpha * diffs_local[:, t] + one_m_a * v; vs[:, t] = v
    vl = vs[:, -1]; ad = vs[:, 1:] - vs[:, :-1]; a = ad[:, 0]
    for t in range(1, T - 1):
        a = beta * ad[:, t] + one_m_b * a
    return vl, a

def _soft_hit_loss(pred, target, thr=0.013012, k=408.348):
    return (1 - torch.sigmoid(-(torch.norm(pred - target, dim=1) - thr) * k)).mean()

def extract_features(X, mean_stats, std_stats, dir_net, heading_mode="3step"):
    device = X.device; p_last = X[:, 10]; diffs = X[:, 1:] - X[:, :-1]
    n1 = diffs[:, 1:].norm(dim=2, keepdim=True) + 1e-8; n2 = diffs[:, :-1].norm(dim=2, keepdim=True) + 1e-8
    cos_t = ((diffs[:, 1:] * diffs[:, :-1]).sum(dim=2, keepdim=True) / (n1 * n2)).clamp(-1, 1)
    theta_seq = torch.acos(cos_t).squeeze(2)
    theta = theta_seq[:, -1:]; theta_mean = theta_seq.mean(1, keepdim=True); theta_std = theta_seq.std(1, keepdim=True)
    theta_vel = theta_seq[:, -1:] - theta_seq[:, -2:-1]
    theta_acc = theta_seq[:, -1:] - 2*theta_seq[:, -2:-1] + theta_seq[:, -3:-2]
    theta_trend = theta_seq[:, -1:] - theta_seq[:, -3:].mean(1, keepdim=True)
    if dir_net is not None:
        speed_seq = diffs.norm(dim=2); state = torch.cat([speed_seq, theta_seq], dim=1)
        if dir_net[0].in_features == 29:
            state = torch.cat([state, diffs[:, :, 2].abs()], dim=1)
        weights = F.softmax(dir_net(state), dim=1); v_sm = (diffs * weights.unsqueeze(2)).sum(dim=1)
    else:
        v_sm = (3*diffs[:, -1] + 2*diffs[:, -2] + diffs[:, -3]) / 6.0 if heading_mode == "3step" else diffs[:, -1]
    fwd = v_sm / (v_sm.norm(dim=1, keepdim=True) + 1e-8)
    up_w = torch.zeros_like(fwd); up_w[:, 2] = 1.0
    up_w[fwd[:, 2].abs() > 0.99] = torch.tensor([0., 1., 0.], device=device)
    right = torch.cross(fwd, up_w, dim=1); right = right / (right.norm(dim=1, keepdim=True) + 1e-8)
    up = torch.cross(right, fwd, dim=1); up = up / (up.norm(dim=1, keepdim=True) + 1e-8)
    R = torch.stack([fwd, right, up], dim=2)
    v_last = diffs[:, -1]; v_prev1 = diffs[:, -2]; speed = v_last.norm(dim=1, keepdim=True)
    a_last = v_last - v_prev1; acc_mag = a_last.norm(dim=1, keepdim=True)
    v_local = torch.matmul(v_last.unsqueeze(1), R).squeeze(1); a_local = torch.matmul(a_last.unsqueeze(1), R).squeeze(1)
    X_local = torch.matmul(X - p_last.unsqueeze(1), R); p_std_local = X_local.std(1); v_local_abs = v_local.abs()
    jerk_g = diffs[:, -1] - 2*diffs[:, -2] + diffs[:, -3]; jerk_l = torch.matmul(jerk_g.unsqueeze(1), R).squeeze(1)
    jerk_mag = jerk_g.norm(dim=1, keepdim=True)
    features = torch.cat([v_local, a_local, speed, acc_mag, theta, theta_mean, theta_std, theta_trend,
                          theta_vel, theta_acc, p_std_local, v_local_abs, jerk_l, jerk_mag], dim=1)
    if mean_stats is None or std_stats is None:
        mean_stats = features.mean(0, keepdim=True); std_stats = features.std(0, keepdim=True) + 1e-8
    return (features - mean_stats) / std_stats, diffs, p_last, theta, theta_mean, theta_std, theta_seq, R, speed, mean_stats, std_stats

**HyperPhysics 모델** — ResBlock / PriorBiasedLinear / Rodrigues 회전 / turn-rate 물리

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(0.15), nn.Linear(dim, dim))
        self.ln = nn.LayerNorm(dim)
    def forward(self, x): return self.ln(x + self.net(x))

class PriorBiasedLinear(nn.Module):
    def __init__(self, in_f, out_f, prior_bias):
        super().__init__(); self.linear = nn.Linear(in_f, out_f)
        self.register_buffer('prior_bias', prior_bias.clone().detach())
        with torch.no_grad(): nn.init.zeros_(self.linear.weight); nn.init.zeros_(self.linear.bias)
    def forward(self, x): return self.linear(x) + self.prior_bias

def rodrigues_rotate(v, w):
    theta = w.norm(dim=1, keepdim=True); k = w / (theta + 1e-8)
    cos_t = torch.cos(theta); sin_t = torch.sin(theta); dot = (v * k).sum(dim=1, keepdim=True); cross = torch.cross(k, v, dim=1)
    return v * cos_t + cross * sin_t + k * dot * (1.0 - cos_t)

class HyperPhysics_xy2(nn.Module):
    def __init__(self, input_dim=24, **kw):
        super().__init__()
        self.sh_thr=kw.pop('sh_thr',0.013012); self.sh_k=kw.pop('sh_k',408.348044); self.mse_w=kw.pop('mse_w',129.172037)
        self.local_w=kw.pop('local_w',0.050941); self.theta_thr=kw.pop('theta_thr',1.087618); self.speed_thr=kw.pop('speed_thr',0.034583)
        self.lr=0.005400; self.wd=0.005659
        self.register_buffer("mean_stats", torch.zeros(1, input_dim)); self.register_buffer("std_stats", torch.ones(1, input_dim))
        self.use_dirnet = True   # False면 고정 3-step heading (다른 프레임 → decorrelated 2nd 물리)
        prior_dir = torch.tensor([-10.,-10.,-10.,-10.,-10.,-10.,-10.,0.,0.693,1.098])
        self.dir_net = nn.Sequential(nn.Linear(29,24), nn.LayerNorm(24), nn.GELU(), PriorBiasedLinear(24,10,prior_dir))
        self.temporal_net = nn.Sequential(nn.Linear(9,32), nn.LayerNorm(32), nn.GELU(), PriorBiasedLinear(32,6,torch.zeros(6)))
        prior_dyn = torch.tensor([0.,0.,0.,0.,0.,0.]+[-4.]*24)
        self.dynamics_net = nn.Sequential(nn.Linear(input_dim,96), nn.LayerNorm(96), nn.GELU(), ResBlock(96), PriorBiasedLinear(96,30,prior_dyn))
        self.omega_w = nn.Parameter(torch.tensor([0.0,-0.5,-1.0]))
        self.omega_net = nn.Sequential(nn.LayerNorm(input_dim), nn.Linear(input_dim,48), nn.GELU(), nn.Linear(48,3))
        with torch.no_grad(): nn.init.normal_(self.omega_net[-1].weight, std=0.01); nn.init.zeros_(self.omega_net[-1].bias)
        self.diffusion_net = nn.Sequential(nn.Linear(input_dim,32), nn.LayerNorm(32), nn.GELU(), nn.Linear(32,3))
    def get_features(self, X, mean_stats=None, std_stats=None):
        return extract_features(X, mean_stats, std_stats, self.dir_net if self.use_dirnet else None, "3step")
    @staticmethod
    def _rotation_vector(d_prev, d_curr):
        n_prev = d_prev.norm(dim=1, keepdim=True).clamp(min=1e-8); n_curr = d_curr.norm(dim=1, keepdim=True).clamp(min=1e-8)
        d_hat_prev = d_prev/n_prev; d_hat_curr = d_curr/n_curr; cross = torch.linalg.cross(d_hat_prev, d_hat_curr, dim=1)
        sin_t = cross.norm(dim=1, keepdim=True).clamp(min=1e-8); cos_t = (d_hat_prev*d_hat_curr).sum(1, keepdim=True).clamp(-0.9999, 0.9999)
        theta = torch.atan2(sin_t, cos_t); speed_gate = torch.sigmoid((n_prev+n_curr)*500-5)
        return cross/sin_t*theta*speed_gate
    def forward(self, features, diffs, p_last, theta, speed, R):
        B = diffs.shape[0]
        ema_raw = self.temporal_net(features[:, 8:17]); alpha = torch.sigmoid(ema_raw[:, 0:3])*0.8+0.1; beta = torch.sigmoid(ema_raw[:, 3:6])*0.199+0.8
        dyn_raw = self.dynamics_net(features); w_v = 2.0+dyn_raw[:, 0:3]; w_a = 1.0+dyn_raw[:, 3:6]
        v_local_abs = features[:, 17:20]; v_local_abs2 = v_local_abs*v_local_abs; theta2 = theta*theta
        exp_v = (F.softplus(dyn_raw[:,6:9])*v_local_abs + F.softplus(dyn_raw[:,9:12])*v_local_abs2 + F.softplus(dyn_raw[:,12:15])*theta + F.softplus(dyn_raw[:,15:18])*theta2)
        exp_a = (F.softplus(dyn_raw[:,18:21])*v_local_abs + F.softplus(dyn_raw[:,21:24])*v_local_abs2 + F.softplus(dyn_raw[:,24:27])*theta + F.softplus(dyn_raw[:,27:30])*theta2)
        diffs_local = torch.matmul(diffs, R); vl, al = _ema_va_local(diffs_local, alpha, beta); diff_speed = diffs_local.norm(dim=2)
        def rv_masked(ka, kb):
            rv = self._rotation_vector(diffs_local[:, ka], diffs_local[:, kb])
            valid = ((diff_speed[:, ka] > 1e-5) & (diff_speed[:, kb] > 1e-5)).float()
            return rv*valid.unsqueeze(1), valid
        ov1, vm1 = rv_masked(-2,-1); ov2, vm2 = rv_masked(-3,-2); ov3, vm3 = rv_masked(-4,-3)
        w_logits = self.omega_w.view(1,3).expand(B,-1); masks = torch.stack([vm1, vm2, vm3], dim=1)
        w_attn = F.softmax(w_logits.masked_fill(masks == 0, -1e9), dim=1)
        omega_hist = w_attn[:,0].unsqueeze(1)*ov1 + w_attn[:,1].unsqueeze(1)*ov2 + w_attn[:,2].unsqueeze(1)*ov3
        current_speed = speed.view(B,1) if speed is not None else diff_speed[:,-1].unsqueeze(1)
        omega_delta = self.omega_net(features) * torch.sigmoid(current_speed*500-5)
        theta_scalar = theta.view(B,1)
        rotation_gate = torch.sigmoid((theta_scalar-self.theta_thr)*10) * torch.sigmoid((current_speed-self.speed_thr)*200)
        omega = (omega_hist + omega_delta) * rotation_gate
        v_rotated = rodrigues_rotate(vl, omega)
        pred_local = (w_v*torch.exp(-exp_v))*v_rotated + (w_a*torch.exp(-exp_a))*al
        log_var = self.diffusion_net(features).clamp(min=-5.0, max=5.0)
        pred_global = p_last + torch.einsum('nij,nj->ni', R, pred_local)
        return pred_global, pred_local, log_var
    def compute_loss(self, pp, yr, pred_local=None, yr_local=None, log_var=None, **kw):
        loss = _soft_hit_loss(pp, yr, self.sh_thr, self.sh_k) + self.mse_w*F.mse_loss(pp, yr)
        if pred_local is not None and yr_local is not None and log_var is not None:
            nll = 0.5*(torch.exp(-log_var)*(pred_local-yr_local)**2 + log_var); loss = loss + self.local_w*nll.mean()
        return loss

**5-fold OOF 학습/추론** — `train_fold` + `predict_full`

In [ ]:
@torch.no_grad()
def predict_full(model, X, device):
    Xt = torch.tensor(X, dtype=torch.float32, device=device); preds = []
    for i in range(0, len(Xt), 512):
        b = Xt[i:i+512]
        ft, df, pl, th, _, _, _, R, sp, _, _ = model.get_features(b, model.mean_stats, model.std_stats)
        pp, _, _ = model(ft, df, pl, th, sp, R); preds.append(pp.cpu().numpy())
    return np.concatenate(preds, 0)

def hit(p, y): return float((np.linalg.norm(p - y, axis=-1) <= 0.01).mean())

def train_fold(Xtr, ytr, epochs, min_win, targets_ext, device, use_dirnet=True):
    ds = SlidingWindowDataset(Xtr, ytr, min_win=min_win, mode="extended", device=device, targets_ext=targets_ext)
    sampler = WeightedRandomSampler(ds.theta_weights, len(ds), replacement=True)
    loader = DataLoader(ds, batch_size=256, sampler=sampler)
    model = HyperPhysics_xy2().to(device); model.use_dirnet = use_dirnet
    with torch.no_grad():
        _, _, _, _, _, _, _, _, _, mn, st = model.get_features(torch.tensor(Xtr, dtype=torch.float32, device=device))
        model.mean_stats.copy_(mn); model.std_stats.copy_(st)
    opt = torch.optim.AdamW(model.parameters(), lr=model.lr, weight_decay=model.wd)
    sch = torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5)
    for ep in range(1, epochs+1):
        model.train(); tl = 0.0
        for Xb, yb in loader:
            opt.zero_grad(set_to_none=True)
            ft, df, pl, th, _, _, _, R, sp, _, _ = model.get_features(Xb, model.mean_stats, model.std_stats)
            pp, prl, lv = model(ft, df, pl, th, sp, R)
            yr_local = torch.matmul((yb - pl).unsqueeze(1), R).squeeze(1)
            loss = model.compute_loss(pp, yb, prl, yr_local, lv)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); tl += loss.item()*len(Xb)
        sch.step()
        if ep <= 3 or ep % 4 == 0: print(f"    ep{ep}/{epochs} loss={tl/len(ds):.4f}", flush=True)
    return model

---
## §6. DE 블렌드(base) + 최종 α 주입 (v157)

**base** = 위 paradigm 멤버들(~40~99개)의 OOF에 `scipy.differential_evolution`으로 softmax 가중을 학습한 conservative 블렌드(OOF 0.6831, 결정론적). **최종**은 여기에 CREE 3-앙상블을 **수동 α로 over-convert** 주입:

```python
cree_ens3 = mean(cree_xy2, cree_xy2s1, cree_xy2h3)        # CREE 회전물리 3-앙상블
v157_a040 = 0.60 * base + 0.40 * cree_ens3                # Public 0.7022
v157_a045 = 0.55 * base + 0.45 * cree_ens3                # Public 0.7022  →  Private 0.703151
```

DE 블렌드의 핵심 로직(가중 학습 + per-axis 보정 + CREE-forward 주입)은 `src/v148_reblend.py`, 최종 제출 생성은 `src/v157_final_submission.py` 입니다. base 재현은 전체 멤버 캐시(`data/cache/*_state.npz`)가 필요하므로, 아래 §7에서는 **frozen된 base + 3-CREE 예측**으로 최종 제출을 즉시 재현·검증합니다.

---
## §7. 빠른 재현 & 검증 (실행 가능, <1초, GPU 불필요)

`submissions/inputs/` 의 frozen 예측(base + 3-CREE)만으로 최종 제출 2개를 재생성하고 원본(`submissions/submission_v157_*.csv`)과 일치하는지 검증합니다. (이 셀은 단독 실행 가능)

In [ ]:
def _xyz(p):
    return pd.read_csv(p)[['x', 'y', 'z']].to_numpy()

INP = ROOT / 'submissions' / 'inputs'
SUB = ROOT / 'submissions'

base = _xyz(INP / 'base_v148blend.csv')
cree = [_xyz(INP / f'cree_{t}.csv') for t in ('xy2', 'xy2s1', 'xy2h3')]
cree_ens3 = np.mean(cree, axis=0)
ids = pd.read_csv(INP / 'base_v148blend.csv')['id']

specs = [(0.40, 'submission_v157_ens3a0.40_FINAL.csv'),
         (0.45, 'submission_v157_ens3a0.45_FINAL.csv')]
for a, fn in specs:
    pred = (1.0 - a) * base + a * cree_ens3
    out = pd.DataFrame({'id': ids, 'x': pred[:, 0], 'y': pred[:, 1], 'z': pred[:, 2]})
    ref_path = SUB / fn
    if ref_path.exists():
        d = np.linalg.norm(pred - _xyz(ref_path), axis=-1).max() * 1000
        print(f'[alpha={a}] {fn}: max diff = {d:.5f} mm  -> {"MATCH" if d < 0.01 else "MISMATCH"}')
    else:
        print(f'[alpha={a}] {fn}: 원본 없음(스킵). 예측 생성만 수행.')
print('\n[done] 최종 제출 = submission_v157_ens3a0.40 / a0.45  (Private 0.703151, 2위)')

---
### 검증된 dead-end (재시도 가치 없음)

| 시도 | 결과 |
|---|---|
| Disagreement selector (per-sample 모델 선택) | DEAD — route-acc 0.17 ≈ 무작위 |
| Mode-seeking / geometric-median 집계 | Δ ≤ 0 (active 멤버 동질 군집) |
| IMM / analytic Constant-Turn 필터 | 0.24~0.55 < naive linear 0.58 |
| Neural CDE (torchcde) | DEAD — OOF 0.2768, 학습 실패 |
| Flow/SONODE 추가 주입 (4-mechanism) | Public 0.6994 < 순수 CREE 0.7022, 폐기 |
| 같은 frenet 프레임 encoder 변종(Transformer/LRU/TCN) | DE weight 0 (포화) |
| pseudo-label | OOF 과적합, LB 변환률 붕괴 |

**교훈**: 점수를 움직인 건 항상 *직교한 새 메커니즘*이었고, 신규 멤버는 **OOF-vs-TEST 예측 L2 일관성**을 필수 검증해야 한다. 자세한 내용은 첨부 PDF(솔루션 문서) 참조.